# Multi-dataset LLR-prior popDMS analysis

This notebook is a thin interactive view over the `esmdms` package. Dataset validation, embeddings, SAE features, inference, sweeps, and metrics live in tested Python modules.

## Prerequisites

The notebook reads canonical datasets and feature artifacts from disk; it does **not** create them. Both directories are git-ignored, so a fresh clone has neither. Run these once from the repository root before executing the cells below:

```bash
# 1. Convert the supplied source tables into canonical datasets/
esmdms process imported_data --output datasets

# 2. Generate the embedding, SAE, and LLR artifacts named in the config
esmdms embed datasets/MV_BRCA1_Findlay_2018 --model biohub/ESMC-300M \
  --layers 30 --pooling max --output artifacts/MV_BRCA1_Findlay_2018/300M
esmdms sae datasets/MV_BRCA1_Findlay_2018 \
  artifacts/MV_BRCA1_Findlay_2018/300M/layer_30.npz \
  --features 1920 --mode batchtopk --k 64 \
  --model-output artifacts/MV_BRCA1_Findlay_2018/300M_layer30_sae.pt \
  --output artifacts/MV_BRCA1_Findlay_2018/300M_layer30_sae.npz
esmdms llr datasets/MV_BRCA1_Findlay_2018 --model biohub/ESMC-300M \
  --output artifacts/MV_BRCA1_Findlay_2018/300M_llr.npz
```

Steps 2 needs the ESM-C weights and a GPU is strongly preferred. `python3 examples/brca1_100_demo.py` runs the same pipeline end to end on a 100-variant subset if you only want something small and runnable.

Add further datasets to `analysis_config.example.json` under `datasets`; the workflow loops over all of them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from esmdms.workflow import load_config, run_analysis

CONFIG_PATH = Path("analysis_config.example.json")
config = load_config(CONFIG_PATH)
config

## Preflight

Fail early with an actionable message instead of a bare `FileNotFoundError` part-way through the run.

In [ ]:
def preflight(config):
    """Return every configured path that does not exist yet."""
    from esmdms.workflow import _resolve

    missing = []
    for spec in config["datasets"]:
        dataset_dir = _resolve(config, spec["path"])
        if not (dataset_dir / "dataset.json").is_file():
            missing.append(f"dataset   {dataset_dir}")
        for group in ("features", "priors"):
            for label, path in spec.get(group, {}).items():
                resolved = _resolve(config, path)
                if not resolved.is_file():
                    missing.append(f"{group[:-1]:9s} {label}: {resolved}")
    return missing


missing = preflight(config)
if missing:
    print("Missing inputs - see the Prerequisites cell above:\n  " + "\n  ".join(missing))
else:
    print("All configured datasets and artifacts are present.")

## Run analysis

`run_analysis` writes `baselines.csv`, `prior_sweeps.csv`, `summary.csv`, and one gamma (or alpha-by-gamma) table per feature artifact and prior into the configured `output_dir`.

In [ ]:
results = run_analysis(config)
summary = results["summary"]
summary

## Baselines

Every gamma-selected row records the `gamma` it used and the cross-replicate consistency that selected it. `gamma` is deliberately `NaN` for methods that do not have one (enrichment ratio, DMS functional score, raw LLR).

`gamma` here is chosen by **unsupervised** cross-replicate consistency, never by ClinVar labels. `spearman_rho` compares inferred fitness to the assay's own functional score and does not depend on the ClinVar review-star cutoff, so it is reported once.

In [ ]:
results["baselines"]

## Best prior configuration by dataset

**Read this before interpreting the chart.** The row below is chosen by *maximum ClinVar AUC* over the whole alpha-by-gamma grid, which is supervised model selection on the same labels used to score it. These numbers are an upper bound, not a generalization estimate; use held-out data for any generalization claim.

`alpha = 0` is in the grid as the regularized zero-mean popDMS control, so the winning configuration may use **no prior at all**. `prior_used` and `auc_gain_over_alpha0` make that explicit, and the bar labels carry the selected alpha.

In [ ]:
if summary.empty:
    print("No prior sweeps were configured.")
else:
    auc_column = f"auc_stars_{int(config.get('review_star_cutoffs', [0])[0])}"
    labels = [
        f"{row.dataset} / {row.prior}\n(alpha={row.alpha:g}, gamma={row.gamma:.3g})"
        for row in summary.itertuples()
    ]
    colors = ["#3A7D44" if used else "#B0B0B0" for used in summary["prior_used"]]

    ax = summary.set_axis(labels).plot.bar(
        y=auc_column, legend=False, figsize=(11, 5), color=colors, width=0.6
    )
    ax.axhline(0.5, color="#C0392B", linestyle="--", linewidth=1, label="chance")
    ax.set_xlabel("")
    ax.set_ylabel("Assay-oriented ClinVar AUC")
    ax.set_ylim(0, 1)
    ax.set_title("Best LLR-prior configuration (grey = winner used no prior)")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")
    plt.tight_layout()

## Does the prior actually help?

The bar above reports one supervised winner. This table asks the more useful question: at each prior strength, what is the best achievable AUC? A prior that helps should beat its own `alpha = 0` row.

In [ ]:
sweeps = results["sweeps"]
if sweeps.empty:
    print("No prior sweeps were configured.")
else:
    auc_column = f"auc_stars_{int(config.get('review_star_cutoffs', [0])[0])}"
    by_alpha = (
        sweeps.pivot_table(
            index=["dataset", "prior"], columns="alpha", values=auc_column, aggfunc="max"
        )
        .rename_axis(columns="alpha (prior strength)")
    )
    display(by_alpha)

    ax = by_alpha.T.plot(marker="o", figsize=(8, 4.5))
    ax.axhline(0.5, color="#C0392B", linestyle="--", linewidth=1)
    ax.set_xlabel("alpha (prior strength; 0 = no prior)")
    ax.set_ylabel("Best assay-oriented ClinVar AUC")
    ax.set_title("Best AUC achievable at each prior strength")
    ax.legend(fontsize=8)
    plt.tight_layout()

## Regularization sensitivity

The full alpha-by-gamma surface behind the single number above.

In [ ]:
if not sweeps.empty:
    auc_column = f"auc_stars_{int(config.get('review_star_cutoffs', [0])[0])}"
    for (dataset_name, prior_name), frame in sweeps.groupby(["dataset", "prior"]):
        ax = (
            frame.pivot_table(index="gamma", columns="alpha", values=auc_column)
            .plot(logx=True, marker=".", figsize=(8, 4))
        )
        ax.axhline(0.5, color="#C0392B", linestyle="--", linewidth=1)
        ax.set_xlabel("gamma (prior precision / regularization)")
        ax.set_ylabel("Assay-oriented ClinVar AUC")
        ax.set_title(f"{dataset_name} / {prior_name}")
        ax.legend(title="alpha", fontsize=8)
        plt.tight_layout()
        plt.show()